

# Question 1 — E-Commerce Order & Inventory Management System

### Difficulty: ⭐⭐⭐⭐⭐ Advanced



### Real-world scenario: Amazon/Flipkart-type backend

You are working as a **Database Developer** for an e-commerce company.

The company currently manages customers, products, warehouses, inventory, orders, payments, shipments, and product reviews using spreadsheets. This is causing problems such as:

* Duplicate customer records
* Incorrect inventory counts
* Products being ordered despite insufficient stock
* Orders containing multiple products
* Partial shipments
* Payment failures
* Customers placing multiple orders
* Difficulty calculating sales and inventory reports

Your task is to **design a complete relational PostgreSQL database** for this system.

---

In [3]:
CREATE DATABASE Amazon_db;

SyntaxError: invalid syntax (560541319.py, line 1)

## Business Requirements

Your database should be capable of handling:



### 1. Customers

Each customer should have:

* Unique customer ID
* Full name
* Email
* Phone number
* Registration date
* City
* State

A customer can place **many orders**.


In [6]:
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    full_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    phone_number VARCHAR(10),
    registration_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    city VARCHAR(50),
    state VARCHAR(50)
)

SyntaxError: invalid syntax (2390659513.py, line 1)


---

### 2. Products

Each product should have:

* Product ID
* Product name
* Category
* Brand
* Price
* Cost price
* Current status (`active`, `inactive`)
* Product creation date

A product belongs to **one category**, but a category can contain many products.


In [ ]:
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    brand VARCHAR(50),
    price NUMERIC(8,2) CHECK(price >= 0),
    cost_price NUMERIC(8,2) CHECK(cost_price >= 0),
    is_active BOOLEAN DEFAULT TRUE,
    product_creation_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)

SyntaxError: invalid syntax (438421431.py, line 1)


---

### 3. Warehouses

The company has multiple warehouses.

Each warehouse should contain:

* Warehouse ID
* Warehouse name
* City
* State

A product can be stored in **multiple warehouses**.

A warehouse can store **many products**.

You therefore need to think carefully about the relationship between **products and warehouses**.

In [ ]:
CREATE TABLE IF NOT EXISTS warehouses (
    warehouse_id INTEGER GENERATED ALWAYS AS INTEGER PRIMARY KEY,
    warehouse_name VARCHAR(100) NOT NULL,
    city VARCHAR(50),
    state VARCHAR(50)
)

---

### 4. Inventory

For every product in every warehouse, the company needs to track:

* Available quantity
* Reserved quantity
* Reorder level
* Last updated timestamp

Example:

A product may have:

> Delhi Warehouse → 100 units
> Agra Warehouse → 40 units
> Lucknow Warehouse → 0 units

In [ ]:
CREATE TABLE IF NOT EXISTS inventory(
    product_id INT NOT NULL,
    warehouse_id INT NOT NULL,
    available_quantity INTEGER CHECK(available_quantity >= 0),
    reserved_quantity INTEGER CHECK(reserved_quantity >= 0),
    reorder_level INTEGER,
    last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(product_id, warehouse_name),
    FOREIGN KEY(product_id) REFERENCES products(product_id),
    FOREIGN KEY(warehouse_id) REFERENCES warehouses(warehouse_id)
)


---

### 5. Orders

A customer can place multiple orders.

An order should contain:

* Order ID
* Customer
* Order date
* Order status
* Shipping address
* Total amount

Possible order statuses:

* `pending`
* `confirmed`
* `processing`
* `shipped`
* `delivered`
* `cancelled`

**Important:** One order can contain **multiple products**.

For example:

**Order #1001**

| Product  | Quantity |
| -------- | -------: |
| Laptop   |        1 |
| Mouse    |        2 |
| Keyboard |        1 |

You need to design the database accordingly.

In [ ]:
CREATE TABLE IF NOT EXISTS orders(
    order_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE DEFAULT CURRENT_DATE,
    order_status VARCHAR(50) CHECK(order_status IN('pending', 'confirmed', 'processing', 'shipped', 'delivered', 'cancelled')),
    shipping_address TEXT NOT NULL,
    total_amount NUMERIC(10,2) CHECK(total_amount >=0),

    FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
)


---

### 6. Order Items

For every product purchased in an order, store:

* Product
* Quantity
* Price at the time of purchase
* Discount

**Important business rule:**

If a product's price changes later, the historical order should **still retain the original purchase price**.

In [ ]:
CREATE TABLE IF NOT EXISTS order_items(
    product_id INT NOT NULL,
    quantity INTEGER CHECK(quantity >= 0),
    purchase_price NUMERIC(10,2) CHECK(purchase_price >= 0),
    discount NUMERIC(8,2) CHECK(discount >= 0),

    PRIMARY KEY(product_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
)





---

### 7. Payments

Each order can have payment information.

Track:

* Payment ID
* Order
* Payment method
* Payment amount
* Payment date
* Payment status
* Transaction/reference ID

Possible payment methods:

* `UPI`
* `Credit Card`
* `Debit Card`
* `Net Banking`
* `Cash on Delivery`

Possible payment statuses:

* `pending`
* `successful`
* `failed`
* `refunded`

In [ ]:
CREATE TABLE IF NOT EXISTS payments(
    payment_id INTEGER GERERATED ALWAYS AS INTEGER,
    transaction_reference_id VARCHAR(100) NOT NULL,
    order_id INT NOT NULL,
    payment_method VARCHAR(30) CHECK(payment_method IN('UPI', 'Credit Card', 'Debit Card', 'Net Banking', 'Cash on Delivery')),
    payment_amount NUMERIC(10,2) CHECK(payment_amount >= 0),
    payment_date DATE DEFAULT CURRENT_DATE,
    payment_status VARCHAR(30) CHECK(payment_status IN('pending', 'successful', 'failed', 'refunded')),

    PRIMARY KEY(payment_id, order_id),
    FOREIGN KEY(order_id) REFERENCES orders(order_id)
)



---

### 8. Shipments

An order may be shipped from one or more warehouses.

For example:

Customer orders:

* Laptop
* Mouse

Laptop may be shipped from Delhi Warehouse while Mouse is shipped from Agra Warehouse.

Therefore, your database should support **partial/multiple shipments for a single order**.

Track:

* Shipment ID
* Order
* Warehouse
* Shipment date
* Delivery date
* Courier name
* Tracking number
* Shipment status

In [ ]:
CREATE TABLE IF NOT EXISTS shipments(
    shipment_id INTEGER GENERATED ALWAYS AS INTEGER PRIMARY KEY,
    order_id INT NOT NULL,
    warehouse_id INT NOT NULL,
    shipment_date TIMESTAMPZ DEFAULT CURRENT_TIMESTAP,
    delivery_date TIMESTAMP,
    courier_name VARCHAR(50) NOT NULL,
    tracking_number VARCHAR(50),
    shipment_status VARCHAR(100),

    PRIMARY KEY(shipment_id, order_id, warehouse_id),
    FOREIGN KEY(order_id) REFERENCES orders(order_id),
    FOREIGN KEY(warehouse_id) REFERENCES warehouses(warehouse_id)
)



---

### 9. Product Reviews

Customers can review products they purchased.

Each review should contain:

* Review ID
* Customer
* Product
* Rating (`1–5`)
* Review text
* Review date

**Business rule:**

A customer should not be able to submit multiple reviews for the same product for the same purchase/order.

In [ ]:
CREATE TABLE IF NOT EXISTS product_reviews(
    review_id INTEGER GENERATED ALWAYS AS INTEGER,
    customer_id INT NOT NULL,
    product_id INT NOT NULL,
    rating INTEGER CHECK(rating IN('1', '2', '3', '4', '5')),
    review_date DATE DEFAULT CURRENT_DATE,

    PRIMARY KEY(review_id, customer_id, product_id),
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
)

---

# Your Tasks

Design the **complete PostgreSQL database** for this e-commerce system.

You need to decide:

1. What tables are required?
2. What columns should each table contain?
3. Which columns should be `PRIMARY KEY`?
4. Which columns should be `FOREIGN KEY`?
5. Which relationships are:

   * One-to-One
   * One-to-Many
   * Many-to-Many
6. Where should you use junction/bridge tables?
7. Which columns should have:

   * `NOT NULL`
   * `UNIQUE`
   * `CHECK`
   * `DEFAULT`
8. How will you prevent invalid inventory quantities?
9. How will you maintain historical product prices in orders?
10. How will you support multiple warehouses for the same product?
11. How will you support multiple shipments for one order?
12. How will you maintain referential integrity?




---

# Interview-Level SQL Problems

After designing the database, you should eventually be able to answer questions like:

### Q1.

Find the **top 5 customers by total spending**.



### Q2.

Find products whose **total inventory across all warehouses is below their reorder level**.



### Q3.

Find the **top 10 best-selling products** based on quantity sold.


### Q4.

Find customers who have placed **more than 3 orders but have never cancelled an order**.



### Q5.

Find the **warehouse with the highest inventory value**.



### Q6.

Find products that are **available in multiple warehouses**.




### Q7.

Find customers who purchased a product but **have not reviewed it**.



### Q8.

Find the percentage of orders that were **cancelled**.

### Q9.

Find the **monthly revenue for the last 12 months**.



### Q10.

Find products whose current price is **higher than their average historical selling price**.


### Q11.

Find orders where the **payment amount doesn't match the order total**.


### Q12.

Find customers whose **average order value is higher than the overall average order value**.


### Q13.

Find the **most profitable products**, considering selling price and cost price.




### Q14.

Find customers who purchased products from **at least 3 different categories**.



### Q15.

Find the warehouse that has fulfilled the **highest number of shipped order items**.



---

# Starter Dataset

I've intentionally kept the dataset **small enough to enter easily but complex enough to create meaningful queries**.

You can insert these records into your tables after designing the schema.

### Customers

```sql
INSERT INTO customers VALUES
(1, 'Aman Sharma', 'aman@gmail.com', '9876543210', '2026-01-10', 'Delhi', 'Delhi'),
(2, 'Riya Singh', 'riya@gmail.com', '9876543211', '2026-01-15', 'Agra', 'Uttar Pradesh'),
(3, 'Rahul Verma', 'rahul@gmail.com', '9876543212', '2026-02-01', 'Lucknow', 'Uttar Pradesh'),
(4, 'Priya Gupta', 'priya@gmail.com', '9876543213', '2026-02-10', 'Jaipur', 'Rajasthan'),
(5, 'Arjun Mehta', 'arjun@gmail.com', '9876543214', '2026-02-20', 'Noida', 'Uttar Pradesh'),
(6, 'Neha Kapoor', 'neha@gmail.com', '9876543215', '2026-03-05', 'Delhi', 'Delhi');
```

### Categories

```sql
INSERT INTO categories VALUES
(1, 'Electronics'),
(2, 'Computers'),
(3, 'Accessories'),
(4, 'Home Appliances');
```

### Products

```sql
INSERT INTO products VALUES
(101, 'Laptop Pro 15', 2, 'TechBrand', 75000, 60000, 'active', '2026-01-01'),
(102, 'Wireless Mouse', 3, 'LogiTech', 1500, 900, 'active', '2026-01-05'),
(103, 'Mechanical Keyboard', 3, 'KeyMaster', 4500, 2800, 'active', '2026-01-08'),
(104, 'Smartphone X', 1, 'TechBrand', 45000, 36000, 'active', '2026-01-12'),
(105, 'Air Purifier', 4, 'CleanAir', 12000, 8500, 'active', '2026-02-01'),
(106, 'USB-C Hub', 3, 'ConnectPro', 2500, 1400, 'active', '2026-02-05'),
(107, 'Gaming Monitor', 2, 'ViewMax', 28000, 21000, 'active', '2026-02-15'),
(108, 'Bluetooth Speaker', 1, 'SoundMax', 5000, 3000, 'active', '2026-03-01');
```

### Warehouses

```sql
INSERT INTO warehouses VALUES
(1, 'Delhi Central Warehouse', 'Delhi', 'Delhi'),
(2, 'Agra Distribution Center', 'Agra', 'Uttar Pradesh'),
(3, 'Lucknow Warehouse', 'Lucknow', 'Uttar Pradesh');
```

### Inventory

```sql
INSERT INTO inventory VALUES
(101, 1, 25, 3, 10, '2026-09-01'),
(101, 2, 10, 2, 8, '2026-09-01'),
(102, 1, 100, 10, 30, '2026-09-01'),
(102, 2, 40, 5, 20, '2026-09-01'),
(103, 1, 35, 5, 15, '2026-09-01'),
(104, 1, 20, 4, 10, '2026-09-01'),
(104, 3, 15, 2, 8, '2026-09-01'),
(105, 2, 8, 1, 10, '2026-09-01'),
(106, 1, 50, 5, 20, '2026-09-01'),
(107, 3, 6, 1, 8, '2026-09-01'),
(108, 2, 25, 3, 10, '2026-09-01');
```

---

## Orders

```sql
INSERT INTO orders VALUES
(1001, 1, '2026-08-01', 'delivered', 'Delhi, Delhi', 76500),
(1002, 2, '2026-08-05', 'delivered', 'Agra, Uttar Pradesh', 49500),
(1003, 3, '2026-08-10', 'shipped', 'Lucknow, Uttar Pradesh', 33000),
(1004, 1, '2026-08-15', 'cancelled', 'Delhi, Delhi', 45000),
(1005, 4, '2026-08-20', 'delivered', 'Jaipur, Rajasthan', 16500),
(1006, 5, '2026-08-25', 'processing', 'Noida, Uttar Pradesh', 77500),
(1007, 6, '2026-09-01', 'confirmed', 'Delhi, Delhi', 5000),
(1008, 2, '2026-09-05', 'delivered', 'Agra, Uttar Pradesh', 30000);
```

---

## Order Items

```sql
INSERT INTO order_items VALUES
(1, 1001, 101, 1, 75000, 0),
(2, 1001, 102, 1, 1500, 0),

(3, 1002, 104, 1, 45000, 0),
(4, 1002, 102, 3, 1500, 500),

(5, 1003, 107, 1, 28000, 0),
(6, 1003, 106, 2, 2500, 0),

(7, 1004, 104, 1, 45000, 0),

(8, 1005, 105, 1, 12000, 0),
(9, 1005, 103, 1, 4500, 0),

(10, 1006, 101, 1, 75000, 0),
(11, 1006, 106, 1, 2500, 0),

(12, 1007, 108, 1, 5000, 0),

(13, 1008, 107, 1, 28000, 0),
(14, 1008, 102, 2, 1500, 1000);
```

---

## Payments

```sql
INSERT INTO payments VALUES
(501, 1001, 'Credit Card', 76500, '2026-08-01', 'successful', 'TXN1001'),
(502, 1002, 'UPI', 49500, '2026-08-05', 'successful', 'TXN1002'),
(503, 1003, 'Debit Card', 33000, '2026-08-10', 'successful', 'TXN1003'),
(504, 1004, 'UPI', 45000, '2026-08-15', 'refunded', 'TXN1004'),
(505, 1005, 'Net Banking', 16500, '2026-08-20', 'successful', 'TXN1005'),
(506, 1006, 'Credit Card', 77500, '2026-08-25', 'successful', 'TXN1006'),
(507, 1007, 'UPI', 5000, '2026-09-01', 'successful', 'TXN1007'),
(508, 1008, 'Cash on Delivery', 30000, '2026-09-05', 'successful', 'TXN1008');
```

---

## Reviews

```sql
INSERT INTO reviews VALUES
(1, 1, 101, 5, 'Excellent laptop.', '2026-08-10'),
(2, 2, 104, 4, 'Good smartphone.', '2026-08-15'),
(3, 4, 105, 5, 'Works very well.', '2026-08-25'),
(4, 2, 102, 4, 'Good mouse.', '2026-08-12'),
(5, 3, 107, 5, 'Excellent display.', '2026-08-20');
```



---

### Your challenge 🔥

**Don't start with SQL queries.**

First design the database on paper:

**Entities → Tables → Attributes → Primary Keys → Foreign Keys → Relationships → Constraints**

Then create the tables yourself.

Once your database is ready, **send me your table design or `CREATE TABLE` statements**. I'll review it like an interviewer—I'll point out normalization issues, wrong relationships, missing constraints, and design mistakes **without solving the case for you**.

Then we'll move to the SQL challenges for this same database.
